<a href="https://colab.research.google.com/github/s-ravi18/Model_Architectures_From_Scratch/blob/main/LLMs/Practice_Attention_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import torch
import torch.functional as F
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

import pandas as pd
import numpy as np


In [85]:
vocab_size = 259
embed_size = 784
max_sequence_length = 10

In [38]:
df = pd.read_csv('sample_text.csv')
df = df.drop(['id'], axis = 1)

In [39]:
df.head(1)

,text,sentiment
0,"I loved the product, it works perfectly.",positive


In [40]:
import pandas as pd
import re

def prepare_text_data(df):

    # Clean and tokenize
    def tokenize(text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        return text.split()

    df["words"] = df["text"].apply(tokenize)

    # Build vocabulary dynamically
    vocab = set()
    for words in df["words"]:
        vocab.update(words)

    # Create word -> ID mapping
    word_to_id = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for word in sorted(vocab):
        word_to_id[word] = len(word_to_id)

    # Convert words -> IDs
    df["input_ids"] = df["words"].apply(
        lambda words: [
            word_to_id.get(word, word_to_id["<UNK>"])
            for word in words
        ]
    )

    # Reverse mapping
    id_to_word = {
        idx: word
        for word, idx in word_to_id.items()
    }

    return df, word_to_id, id_to_word

In [41]:
def positional_encoding(max_sequence_length, embed_size):

  temp = torch.arange(embed_size//2, dtype = torch.float32)
  pos = torch.arange(max_sequence_length).unsqueeze(1)

  final_positional_embedding = torch.zeros(max_sequence_length, embed_size)

  final_positional_embedding[:,0::2] = pos * torch.sin(1 / (1000 ** (2 * temp / embed_size)) )
  final_positional_embedding[:,1::2] = pos * torch.cos(1 / (1000 ** (2 * temp / embed_size)) )

  return final_positional_embedding

In [72]:
def embedding_generation(X, vocab_size, embed_size):
  embed_layer_1 = nn.Embedding(vocab_size + 2, embed_size)
  # Since X is already a tensor (input_ids), we pass it directly to the layer
  return embed_layer_1(X)

In [46]:
df_tokenised, word_to_id, id_to_word = prepare_text_data(df)

In [55]:
df_tokenised.head(1)

,text,sentiment,words,input_ids
0,"I loved the product, it works perfectly.",positive,"[i, loved, the, product, it, works, perfectly]","[119, 141, 230, 186, 132, 258, 175]"


In [58]:
# Extract the token ID lists
sequences = df_tokenised.input_ids.tolist()
# Convert each list to a tensor
sequences = [torch.tensor(seq, dtype=torch.long) for seq in sequences]

# Pad all sequences to the same length
input_ids = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=0
)

In [93]:
input_ids.shape

torch.Size([100, 10])

In [73]:
## converting to embeddings;
df_emb = embedding_generation(input_ids, vocab_size, embed_size)

## positional embedding;
X_pos = positional_encoding(max_sequence_length, embed_size)

## final embedding;
df_pos_embedding = df_emb + X_pos

In [90]:
## 100 inputs, each input is of max_sequence_length = 10 and
## each token is of dimension 784

df_pos_embedding.shape

torch.Size([100, 10, 784])

In [113]:
## creating a single head architecture for attention calculation;
def attention_calculation(X):

  ## weight initialisation;
  Wq, Wv, Wk = torch.randn(embed_size, embed_size), \
                torch.randn(embed_size, embed_size), \
                  torch.randn(embed_size, embed_size)   ## (784, 784)

  ## Q, K, V vectors;

  Q = X @ Wq
  K = X @ Wk
  V = X @ Wv

  ## attention scores and softmax;
  attention_scores = (Q @ K.permute(0, 2, 1)) / (embed_size**0.5)  ## 100, 10, 10
  attention_weights = torch.softmax(attention_scores, dim=-1)

  ## final transformed vector;
  V_new = attention_weights @ V

  return V_new



In [114]:
attention_v1 = attention_calculation(df_pos_embedding)

In [115]:
attention_v1.shape

torch.Size([100, 10, 784])